In [1]:
import pandas as pd
import numpy as np

# PIL

In [2]:
df_pil_pp = pd.read_excel("../DataSets/pil_capite_ch.xlsx")
#drop ultima colonna
df_pil_pp = df_pil_pp.drop(columns=df_pil_pp.columns[-1])
#rinomina le colonne
df_pil_pp.columns = [ 'Year', 'PIL_per_capita_ch', 'percent_change' ]
#rimuovi riga 1 e 2
df_pil_pp = df_pil_pp.drop(index=[0, 1])
#correggi indici
df_pil_pp = df_pil_pp.reset_index(drop=True)
df_pil_pp.head()


,Year,PIL_per_capita_ch,percent_change
0,1991,57310.478997,NaN
1,1992,57917.137264,1.058547
2,1993,58610.126262,1.196518
3,1994,59441.618691,1.418684
4,1995,59688.145398,0.414738


# IPC

In [3]:
df_ipc_annuo = pd.read_excel("../DataSets/ipc_ch.xlsx", sheet_name="INDEX_y")
df_ipc_percent = pd.read_excel("../DataSets/ipc_ch.xlsx", sheet_name="VAR_y-1")

df_ipc_annuo.head()
#rimuovi tutto tranne riga 2,3
df_ipc_annuo = df_ipc_annuo.iloc[2:4, :]
#correggi indici
df_ipc_annuo = df_ipc_annuo.reset_index(drop=True)
df_ipc_annuo.head()

#drop tutte le colonne fino unnamed: 13
df_ipc_annuo = df_ipc_annuo.drop(columns=df_ipc_annuo.columns[0:14])
df_ipc_annuo.head()
#struttura deve avere solo 2 colonne: Year, IPC_index, riga 0 colonna 0 anni e riga 1 colonna  1 IPC index
df_ipc_annuo = df_ipc_annuo.transpose()
df_ipc_annuo.columns = ['Year', 'IPC_index']
df_ipc_annuo = df_ipc_annuo.reset_index(drop=True)
df_ipc_annuo.head()

#ripeti operazioni per df_ipc_percent
df_ipc_percent = df_ipc_percent.iloc[2:4, :]
df_ipc_percent = df_ipc_percent.reset_index(drop=True)
df_ipc_percent = df_ipc_percent.drop(columns=df_ipc_percent.columns[0:14])
df_ipc_percent = df_ipc_percent.transpose()
df_ipc_percent.columns = ['Year', 'IPC_percent_change']
df_ipc_percent = df_ipc_percent.reset_index(drop=True)
df_ipc_percent.head()

#unisci i due dataframe df_ipc_annuo e df_ipc_percent in base alla colonna Year chiamato df_ipc
df_ipc = pd.merge(df_ipc_annuo, df_ipc_percent, on='Year')
df_ipc.head()





,Year,IPC_index,IPC_percent_change
0,1983,63.8,NaN
1,1984,65.7,2.9
2,1985,68,3.4
3,1986,68.5,0.8
4,1987,69.5,1.4


# Unemployment

In [ ]:
all_sheets = pd.read_excel("../DataSets/disoccupazione_ch.xlsx", sheet_name=None)

rows = []

for sheet_name, df_temp in all_sheets.items():
    df_temp.columns = df_temp.iloc[2]
    df_temp = df_temp.drop(index=[0, 1, 2]).reset_index(drop=True)

    # trova la riga "Total" nella prima colonna (adatta la condizione se serve)
    mask = df_temp.iloc[:, 0].astype(str).str.contains("total", case=False, na=False)
    total_row = df_temp.loc[mask].iloc[0]

    total_unemployment = pd.to_numeric(total_row.iloc[1:13], errors="coerce").mean()
    total_unemployment = round(float(total_unemployment), 2)

    rows.append({"Year": sheet_name, "Total_unemployment": total_unemployment})

df_unemployment = pd.DataFrame(rows)

In [29]:
#ordina al contrario in base alla colonna Year
df_unemployment = df_unemployment.sort_values(by='Year', ascending=True).reset_index(drop=True)
#aggiungi colonna increase in base alla colonna Total_unemployment
df_unemployment['Increase'] = df_unemployment['Total_unemployment'].diff().round(2)

In [30]:
df_unemployment.head()

,Year,Total_unemployment,Increase
0,1973,81.00,NaN
1,1974,220.67,139.67
2,1975,10169.75,9949.08
3,1976,20702.50,10532.75
4,1977,12020.42,-8682.08


# Interest rate

In [62]:
df_libor = pd.read_csv("../DataSets/libor_rate.csv", sep=";", skiprows=3)  # o 2,3...
df_tassi = pd.read_csv("../DataSets/tassi_bns+uk+eu.csv", sep=";", skiprows=3)


In [63]:

#elimina righe con L3MCHF in colonna 2
df_libor = df_libor[df_libor.iloc[:, 1] != 'L3MCHF']
#reset index
df_libor = df_libor.reset_index(drop=True)
df_libor.head()


,Date,D0,Value
0,2000-01-03,UG,1.25
1,2000-01-03,OG,2.25
2,2000-01-04,UG,1.25
3,2000-01-04,OG,2.25
4,2000-01-05,UG,1.25


In [64]:

# tipi
df_libor["Date"]  = pd.to_datetime(df_libor["Date"], errors="coerce")
df_libor["Value"] = pd.to_numeric(df_libor["Value"], errors="coerce")

# tieni solo UG/OG
df_uo = df_libor[df_libor["D0"].isin(["UG", "OG"])].dropna(subset=["Date", "Value"])

# 1) (eventuale) media per (Date, D0) se ci sono duplicati nello stesso giorno
daily_uo = (
    df_uo.groupby(["Date", "D0"])["Value"]
         .mean()
         .unstack("D0")   # colonne UG, OG
         .sort_index()
)

# 2) calcolo (UG + OG) / 2 per ogni giorno
daily_uo["UGOG_mean"] = daily_uo[["UG", "OG"]].mean(axis=1)

# (opzionale) tieni solo la colonna finale
daily_mean = daily_uo[["UGOG_mean"]]

# 3) media mensile di quel valore giornaliero
monthly_mean = daily_mean.resample("MS").mean()   # oppure "M" se vuoi fine mese

monthly_mean.head()


D0,UGOG_mean
Date,
2000-01-01,1.750000
2000-02-01,2.202381
2000-03-01,2.478261
2000-04-01,3.000000
2000-05-01,3.000000


In [65]:
df_tassi.head()
#eliminare riche con NaN in colonna 3
#df_tassi = df_tassi.dropna(subset=[df_tassi.columns[2]])
#eliminare righe con H, SRF
df_tassi = df_tassi[~df_tassi.iloc[:, 1].isin(['H', 'SRF'])]
df_tassi.head()


,Date,D0,Value
0,2000-01,LZ,NaN
3,2000-01,EF,2.00
4,2000-01,L0,5.75
5,2000-02,LZ,NaN
8,2000-02,EF,2.25


In [69]:
# 1) monthly_mean -> colonna "month" per fare merge
ch_monthly = (
    monthly_mean
      .reset_index()
      .rename(columns={monthly_mean.index.name or 'Date': 'month'})  # gestisce index senza nome
)

# se dopo reset_index la colonna non si chiama 'month', rinomina la prima colonna in 'month'
if "month" not in ch_monthly.columns:
    ch_monthly = ch_monthly.rename(columns={ch_monthly.columns[0]: "month"})

# 2) df_tassi -> wide (EF, L0, LZ in colonne)
tassi_wide = (
    df_tassi
      .pivot_table(index="Date", columns="D0", values="Value", aggfunc="mean")
      .reset_index()
      .rename(columns={"Date": "month"})
)

# forza stesso tipo per la chiave di merge: mese (Period)
tassi_wide["month"] = pd.to_datetime(tassi_wide["month"]).dt.to_period("M")
ch_monthly["month"] = pd.to_datetime(ch_monthly["month"]).dt.to_period("M")


# 3) merge + CH fill: LZ se c’è, altrimenti UGOG_mean
out = tassi_wide.merge(ch_monthly[["month", "UGOG_mean"]], on="month", how="left")

out["CH_rate"] = out["LZ"].fillna(out["UGOG_mean"])

# 4) rinomina e tieni solo quello che serve
out = (
    out.rename(columns={
        "EF": "EU_rate",
        "L0": "UK_rate",
    })
    .loc[:, ["month", "CH_rate", "EU_rate", "UK_rate"]]
    .sort_values("month")
    .reset_index(drop=True)
)

out.head()

D0,month,CH_rate,EU_rate,UK_rate
0,2000-01,1.750000,2.00,5.75
1,2000-02,2.202381,2.25,6.00
2,2000-03,2.478261,2.50,6.00
3,2000-04,3.000000,2.75,6.00
4,2000-05,3.000000,2.75,6.00
